In [4]:
import torch
print(torch.__version__)

2.6.0+cu124


In [2]:
!pip install huggingface huggingface_hub transformers numpy --quiet


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [3]:
!pip install pillow requests --quiet


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
from huggingface_hub import login
login(token="")

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
!pip install --upgrade transformers accelerate bitsandbytes peft --quiet


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
!apt update && apt install unzip -y

In [ ]:
!unzip Brain_Cancer.zip -d dataset

In [6]:
!pip install datasets --quiet


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
from datasets import load_dataset

train_size = 5000  # @param {type: "number"}
validation_size = 1056  # @param {type: "number"}

data = load_dataset("./dataset/Brain_Cancer", split="train")
data = data.train_test_split(
    train_size=train_size,
    test_size=validation_size,
    shuffle=True,
    seed=42,
)
# Use the test split as the validation set
data["validation"] = data.pop("test")

# Display dataset details
data

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 5000
    })
    validation: Dataset({
        features: ['image', 'label'],
        num_rows: 1056
    })
})

In [ ]:
from typing import Any

TUMOR_CLASSES = [
    "A: glioma",
    "B: menin",
    "C: pitutary"]

options = "\n".join(TUMOR_CLASSES)
PROMPT = f"What is the most likely tumor type shown in the mri image?\n{options}"


def format_data(example: dict[str, Any]) -> dict[str, Any]:
    example["messages"] = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                },
                {
                    "type": "text",
                    "text": PROMPT,
                },
            ],
        },
        {
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": TUMOR_CLASSES[example["label"]],
                },
            ],
        },
    ]
    return example


In [ ]:
from typing import Any

from datasets import load_dataset


def format_test_data(example: dict[str, Any]) -> dict[str, Any]:
    example["messages"] = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                },
                {
                    "type": "text",
                    "text": PROMPT,
                },
            ],
        },
    ]
    return example


test_data = data["validation"]#load_dataset("./Brain_Cancer", split="test")
test_data = test_data.shuffle(seed=42).select(range(1000))
test_data = test_data.map(format_test_data)

In [ ]:
from datasets import ClassLabel

# Rename the class names to the tissue classes, `X: tissue type`
test_data = test_data.cast_column(
    "label",
    ClassLabel(names=TUMOR_CLASSES)
)

LABEL_FEATURE = test_data.features["label"]
# Mapping to alternative label format, `(X) tissue type`
ALT_LABELS = dict([
    (label, f"({label.replace(': ', ') ')}") for label in TUMOR_CLASSES
])


def postprocess(prediction: list[dict[str, str]], do_full_match: bool=False) -> int:
    response_text = prediction[0]["generated_text"]
    if do_full_match:
        return LABEL_FEATURE.str2int(response_text)
    for label in TUMOR_CLASSES:
        # Search for `X: tissue type` or `(X) tissue type` in the response
        if label in response_text or ALT_LABELS[label] in response_text:
            return LABEL_FEATURE.str2int(label)
    return -1

In [ ]:
batch_messages = []

for text, img in zip(test_data["messages"], test_data["image"]):
    batch_messages.append([
        {
            "role": "user",
            "content": [
                {"type": "text", "text": text},
                {"type": "image", "image": img},
            ],
        }
    ])


In [1]:
from transformers import pipeline, AutoModelForImageTextToText, AutoProcessor
from PIL import Image
import requests
import torch
from peft import PeftModel

import transformers.integrations.peft

# --- WORKAROUND: Bypass the PEFT MoE conversion bug ---
if not hasattr(transformers.integrations.peft, "_MOE_TARGET_MODULE_MAPPING"):
    transformers.integrations.peft._MOE_TARGET_MODULE_MAPPING = {}
transformers.integrations.peft._MOE_TARGET_MODULE_MAPPING['llava'] = {}
# ------------------------------------------------------

base_model_id = "google/medgemma-4b-it"
lora_adapter_path = "Hrushikesh-0000/medgemma-4b-it-sft-lora-MRI6k"

processor = AutoProcessor.from_pretrained(base_model_id)

pipe = pipeline(
    "image-text-to-text",
    model=lora_adapter_path,
    processor=processor,
    device="cuda",
    # Note: We omit device="cuda" here because device_map="cuda" handled it during model loading
    dtype=torch.bfloat16,
)

# 2. Apply the exact same generation configs from your eval
pipe.model.generation_config.do_sample = False
pipe.model.generation_config.pad_token_id = processor.tokenizer.eos_token_id
processor.tokenizer.padding_side = "left"

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 800/800 [00:00<00:00, 4841.48it/s]
Gemma3ForConditionalGeneration LOAD REPORT from: Hrushikesh-0000/medgemma-4b-it-sft-lora-MRI6k
Key                                                              | Status     | 
-----------------------------------------------------------------+------------+-
model.language_model.embed_tokens.weight                         | UNEXPECTED | 
base_model.model.lm_head.weight                                  | UNEXPECTED | 
lm_head.modules_to_save.default.weight                           | MISSING    | 
model.language_model.embed_tokens.modules_to_save.default.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect 

In [ ]:
print(batch_messages[0])

[{'role': 'user', 'content': [{'type': 'text', 'text': [{'content': [{'type': 'image'}, {'type': 'text', 'text': 'What is the most likely tumor type shown in the mri image?\nA: glioma\nB: menin\nC: pitutary'}], 'role': 'user'}]}, {'type': 'image', 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=512x512 at 0x7FC26C300460>}]}]


In [ ]:
from transformers.utils import logging
logging.set_verbosity_error()
prompt = "What is the most likely tumor type shown in the mri image, answer in a sentence?\nA: glioma\nB: menin\nC: pitutary"
image = Image.open("./brain_menin_0018.jpg").convert("RGB")
#image = Image.open("./brain_tumor_0007.jpg").convert("RGB")

messages=[
    
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image", "image": image},
            ],
        }
    
]
#"<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=512x512 at 0x7FC26C300460>"
ft_outputs = pipe(
    #batch_messages,
    messages,
    max_new_tokens=1000,
    batch_size=64,
    return_full_text=True,

)

#ft_predictions = [postprocess(out) for out in ft_outputs]
#print(ft_outputs)
print(ft_outputs[0]['generated_text'][-1])

{'role': 'assistant', 'content': 'B: menin'}


: 

In [10]:
!pip install numpy --quiet


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [14]:
import numpy as np
print(np.__version__)

2.2.6


In [ ]:
print(ft_outputs[0]['generated_text'][-1])

{'role': 'assistant', 'content': 'C: pitutary'}


In [ ]:
# Loop through the batch
for output in ft_outputs:
    # Extract the answer (handling the nested list)
    answer = output[0]["generated_text"][-1]["content"]

    # Print the answer, but end with a separator instead of a new line
    print(answer, end="\n")

# Optional: Add a final empty print() to move to a new line when the loop finishes
print()

In [2]:
#del model
del pipe
torch.cuda.empty_cache()

NameError: name 'pipe' is not defined